# 7. 모델 활용 방안 및 한계

서울시 따릉이 일별 수요 예측. [요구사항](../../docs/ml/REQUIREMENTS.md).

In [1]:
from pathlib import Path
import sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from src.ml.features import FEATURES, load_daily, make_features
from src.ml.splits import split_data, time_cv
from src.ml.models import model_candidates
from src.ml.evaluation import metrics, compare, grouped_metrics
from src.ml.tuning import tune_models
from src.ml.predict import forecast
from sklearn.base import clone
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.figsize": (10, 4), "font.family": "Malgun Gothic", "axes.unicode_minus": False})
OUT = ROOT / "outputs/ml"
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = ROOT / "data/processed"
LOCAL.mkdir(parents=True, exist_ok=True)
daily = load_daily(ROOT)
frame = make_features(daily)
train, validation, final_train, test = split_data(frame)


In [2]:
model = joblib.load(LOCAL / "ml_final_pipeline.joblib")
selection = json.loads((OUT / "05_selection.json").read_text(encoding="utf-8"))
mae = pd.read_csv(OUT / "06_test_metrics.csv", index_col=0).loc[selection["model"], "MAE"]
saved = pd.read_csv(LOCAL / "ml_test_predictions.csv", parse_dates=["date"])
# Typical case: the test day whose absolute error is closest to the median.
typical = saved.iloc[(saved.absolute_error - saved.absolute_error.median()).abs().argmin()]
example = forecast(daily, typical.date, model)
assert np.isclose(example.prediction.iloc[0], typical.prediction), "forecast() must match the 6번 evaluation path"
example["actual"] = typical.rentals
example["error"] = example.actual - example.prediction
display(example.T)


,0
date,2026-01-21 00:00:00
lag_2,48086.0
lag_7,75687.0
lag_14,70011.0
rolling_7,60040.0
rolling_28,53352.392857
dow_sin,0.974928
dow_cos,-0.222521
year_sin,0.337523
year_cos,0.941317


- 2026-01-21 입력 10개로 62,913건을 예측했으며 실제 47,549건, 오차 -15,364건임.
- 테스트 절대오차가 중앙값에 가장 가까운 날을 골라 전형적인 사례로 제시함.
- 활용 함수의 예측이 6번 평가의 저장 예측과 같아 운영 입력과 평가 입력이 동일함을 확인함.
- 입력은 D-2까지의 대여건수와 D일 달력뿐이며 D-1 이후 값은 사용하지 않음.

In [3]:
future = pd.concat([forecast(daily, date, model) for date in ["2026-07-01", "2026-07-02"]], ignore_index=True)
future["lower"] = (future.prediction - mae).clip(lower=0)
future["upper"] = future.prediction + mae
display(future.T)
try:
    forecast(daily, "2026-07-03", model)
    unavailable = ""
except ValueError as error:
    unavailable = str(error)
    print(f"2026-07-03 예측 불가: {unavailable}")
examples = pd.concat([example.assign(kind="test"), future.assign(kind="future")], ignore_index=True)
examples.insert(1, "weekday", examples.date.dt.day_name())
examples.to_csv(OUT / "07_examples.csv", index=False)
context = {"model": selection["model"], "test_MAE": mae, "data_end": f"{daily.date.max():%Y-%m-%d}",
           "july_2025_mean": daily.loc[daily.date.dt.strftime("%Y-%m") == "2025-07", "rentals"].mean(),
           "unavailable_2026_07_03": unavailable}
(OUT / "07_context.json").write_text(json.dumps(context, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(context)


,0,1
date,2026-07-01 00:00:00,2026-07-02 00:00:00
lag_2,149167.0,153728.0
lag_7,157582.0,138827.0
lag_14,151025.0,148829.0
rolling_7,141511.428571,141559.571429
rolling_28,134466.428571,134633.642857
dow_sin,0.974928,0.433884
dow_cos,-0.222521,-0.900969
year_sin,0.025818,0.008607
year_cos,-0.999667,-0.999963


2026-07-03 예측 불가: Targets through 2026-07-01 are required


{'model': 'random_forest',
 'test_MAE': np.float64(18370.258006201308),
 'data_end': '2026-06-30',
 'july_2025_mean': np.float64(114071.06451612903),
 'unavailable_2026_07_03': 'Targets through 2026-07-01 are required'}

- 2026-07-01(Wednesday) 예측 141,378건, 참고 범위 123,008~159,749건임.
- 2026-07-02(Thursday) 예측 127,307건, 참고 범위 108,937~145,677건임.
- 6번에서 고정한 최종 모델(2025-12-30까지 학습)을 재학습 없이 그대로 사용함.
- 예측 평균이 2025년 7월 일평균 114,071건보다 +17.8% 차이 남. 6월 말의 높은 수준이 지연값에 반영된 결과로 보이며, 장마 등 7월 날씨 변화는 입력에 없음.
- 참고 범위는 테스트 MAE를 더하고 뺀 값으로 확률적 신뢰구간이 아님.
- 자료가 2026-06-30에 끝나 2026-07-03부터는 D-2 실제값이 없어 예측을 거부함.
- 반기 공개자료로는 매일 예측을 갱신할 수 없어 실제 운영에는 일 집계 공급이 필요함.

In [4]:
months = pd.read_csv(OUT / "06_eda_months.csv", index_col=0)
bands = pd.read_csv(OUT / "06_demand_band.csv", index_col=0)
usage = pd.DataFrame([
    ["입력", "D-2까지 확정된 일별 총 대여건수(최소 28일)와 D일 날짜"],
    ["출력", f"D일 서울시 전체 총 대여건수 1개 값, 테스트 MAE ±{mae:,.0f}건을 참고 범위로 함께 제시"],
    ["활용", "전일 기준 정비·재배치 인력과 차량 규모 계획, 수요 급증·급감 예상일 사전 점검"],
    ["적용 범위", "서울시 전체 일 단위 총량. 대여소·시간대별 배치 결정에는 적용 불가"],
    ["한계", f"날씨·공휴일·운영 변화 미반영, 월별 MAE 최대 {months.MAE.max():,.0f}건, 저수요 구간 WAPE {bands.loc['low', 'WAPE']:.1%}"],
    ["한계", "약 1년 학습, 반기 공개자료라 실시간 갱신 불가, 테스트 기간을 선행 EDA에서 이미 관찰"],
    ["개선", "일 집계 자동 공급, 공휴일 달력, 전일 발행 기상예보, 대여소별 수요·재고 자료, 정기 재학습"],
], columns=["항목", "내용"])
usage.to_csv(OUT / "07_usage.csv", index=False)
display(usage)


,항목,내용
0,입력,D-2까지 확정된 일별 총 대여건수(최소 28일)와 D일 날짜
1,출력,"D일 서울시 전체 총 대여건수 1개 값, 테스트 MAE ±18,370건을 참고 범위..."
2,활용,"전일 기준 정비·재배치 인력과 차량 규모 계획, 수요 급증·급감 예상일 사전 점검"
3,적용 범위,서울시 전체 일 단위 총량. 대여소·시간대별 배치 결정에는 적용 불가
4,한계,"날씨·공휴일·운영 변화 미반영, 월별 MAE 최대 27,341건, 저수요 구간 WA..."
5,한계,"약 1년 학습, 반기 공개자료라 실시간 갱신 불가, 테스트 기간을 선행 EDA에서 ..."
6,개선,"일 집계 자동 공급, 공휴일 달력, 전일 발행 기상예보, 대여소별 수요·재고 자료,..."


- 매일 D-1 00:00에 다음 날 서울시 전체 수요 1개 값을 출력해 운영 규모 계획의 참고 지표로 사용함.
- 예측값은 오차 범위와 함께 제시하며 단독 의사결정 근거로 사용하지 않음.
- 개별 대여소 재배치·시간대 운영에는 대여소별 수요·재고 자료와 별도 모델이 필요함.
- 관측 날씨를 쓴 4회차 사후 실험의 MAE 12,383건과 현재 18,370건의 차이로 볼 때 기상예보 결합이 우선 개선 과제임.
- 공휴일은 사전에 알 수 있어 누수 없이 추가 가능하므로 8번 추가 실험 후보로 둠.